# Build the SqueakView YOLO26 Pose Engine

This notebook is self-contained. It reads the dataset YAML, exports the pose checkpoint directly with Ultralytics, converts the Ultralytics engine into the raw TensorRT plan expected by DeepStream, and writes the model package.

In [1]:
from pathlib import Path
import subprocess
import atexit
import hashlib
import json
import os
import shutil
import sys

import onnx
import tensorrt as trt
import torch
import ultralytics
import yaml
from ultralytics import YOLO

WORKSPACE = Path.cwd().resolve()
if WORKSPACE.name in {"build_engine", "build-engine"}:
    WORKSPACE = WORKSPACE.parent
if str(WORKSPACE) not in sys.path:
    sys.path.insert(0, str(WORKSPACE))
from squeakview.model_build import (
    cleanup_staging_package, create_staging_package, promote_model_package,
    run_trtexec_validation,
)
from squeakview.model_package import validate_model_package

SOURCE_MODEL = WORKSPACE / "build_me/mousehouse_model/mousehouse_best.pt"
DATA_YAML = WORKSPACE / "build_me/mousehouse_model/mousehouse_best.yaml"
PARSER_LIBRARY = WORKSPACE / "native/nvdsinfer_custom_impl_yolo/libnvdsinfer_custom_impl_Yolo.so"
MODEL_NAME = os.environ.get("SQUEAKVIEW_BUILD_MODEL_NAME", "mousehouse").strip()
assert MODEL_NAME and Path(MODEL_NAME).name == MODEL_NAME, "Model name must be one path component"
MODELS_DIR = WORKSPACE / "models"
FINAL_PACKAGE_DIR = MODELS_DIR / MODEL_NAME

PRECISION = "fp16"
BATCH_SIZE = 1
IMAGE_SIZE = 640
DEVICE = 0
CONFIDENCE_THRESHOLD = 0.25
KEYPOINT_THRESHOLD = 0.50
overwrite_raw = os.environ.get("SQUEAKVIEW_BUILD_OVERWRITE", "0")
assert overwrite_raw in {"0", "1"}, "SQUEAKVIEW_BUILD_OVERWRITE must be 0 or 1"
OVERWRITE_EXISTING = overwrite_raw == "1"

assert SOURCE_MODEL.is_file(), SOURCE_MODEL
assert DATA_YAML.is_file(), DATA_YAML
assert PARSER_LIBRARY.is_file(), PARSER_LIBRARY
assert torch.cuda.is_available(), "CUDA is required for TensorRT export"

print(f"Ultralytics {ultralytics.__version__}")
print(f"PyTorch {torch.__version__}")
print(f"TensorRT {trt.__version__}")
print(f"GPU {torch.cuda.get_device_name(DEVICE)}")


Ultralytics 8.4.152
PyTorch 2.14.0+cu130
TensorRT 10.16.0.72
GPU NVIDIA GeForce RTX 4060 Ti


In [2]:
# Derive the complete runtime label/index contract from the YAML and checkpoint.
data_config = yaml.safe_load(DATA_YAML.read_text()) or {}
model = YOLO(str(SOURCE_MODEL))
head = model.model.model[-1]

def ordered_labels(value):
    if isinstance(value, dict):
        def sort_key(key):
            return (0, int(key)) if str(key).isdigit() else (1, str(key))
        return [str(value[key]) for key in sorted(value, key=sort_key)]
    if isinstance(value, (list, tuple)):
        return [str(item) for item in value]
    return []

def class_value(mapping, class_id, class_name):
    if not isinstance(mapping, dict):
        return None
    for key in (class_id, str(class_id), class_name):
        if key in mapping:
            return mapping[key]
    return None

checkpoint_classes = [str(model.names[index]) for index in sorted(model.names)]
class_names = ordered_labels(data_config.get("names")) or checkpoint_classes
checkpoint_kpt_shape = [int(value) for value in head.kpt_shape]
yaml_kpt_shape = data_config.get("kpt_shape")
keypoint_count, keypoint_dims = map(int, yaml_kpt_shape or checkpoint_kpt_shape)
assert model.task == "pose", f"Expected pose, found {model.task}"
one2one_head = getattr(head, "one2one", None)
assert isinstance(one2one_head, dict) and one2one_head, (
    "Expected a YOLO26 pose checkpoint with a selectable one-to-one head"
)
assert checkpoint_classes == class_names, (checkpoint_classes, class_names)
assert checkpoint_kpt_shape == [keypoint_count, keypoint_dims]
assert keypoint_dims == 3, "Pose keypoints must use x/y/confidence"

def global_keypoint_names():
    sources = [
        ("yaml.kp_names", data_config.get("kp_names")),
        ("yaml.keypoint_names", data_config.get("keypoint_names")),
        ("yaml.kpt_names", data_config.get("kpt_names")),
        ("checkpoint.kpt_names", getattr(model, "kpt_names", None)),
    ]
    for source_name, value in sources:
        candidates = []
        if isinstance(value, (list, tuple)):
            candidates = [[str(item) for item in value]]
        elif isinstance(value, dict):
            for class_id, class_name in enumerate(class_names):
                candidate = class_value(value, class_id, class_name)
                if isinstance(candidate, (list, tuple)):
                    candidates.append([str(item) for item in candidate])
        valid = [candidate for candidate in candidates if len(candidate) == keypoint_count]
        if valid and all(candidate == valid[0] for candidate in valid):
            # Numeric checkpoint labels are placeholders; keep searching for YAML names.
            if source_name.startswith("checkpoint") and all(name.isdigit() for name in valid[0]):
                continue
            return valid[0], source_name
    return [f"keypoint_{index}" for index in range(keypoint_count)], "generated"

keypoint_names, keypoint_name_source = global_keypoint_names()
assert len(keypoint_names) == keypoint_count
assert len(set(keypoint_names)) == keypoint_count, "Keypoint names must be unique"
pose_classes = [
    {
        "id": class_id,
        "name": class_name,
        "threshold": CONFIDENCE_THRESHOLD,
        "track": class_id == 0,
        "keypoint_indices": list(range(keypoint_count)),
    }
    for class_id, class_name in enumerate(class_names)
]

EXPECTED_OUTPUT = [BATCH_SIZE, 300, 6 + keypoint_count * keypoint_dims]
print("classes:", class_names)
print(f"keypoints ({keypoint_name_source}):", keypoint_names)
print("class policies:", pose_classes)
print("expected output:", EXPECTED_OUTPUT)


classes: ['mouse', 'landmarks']
keypoints (yaml.kp_names): ['nose', 'head', 'left_ear', 'right_ear', 'back', 'tail_base', 'poke_left', 'well', 'poke_right', 'sipper_left', 'sipper_right', 'top_left_corner', 'top_right_corner', 'bottom_left_corner', 'bottom_right_corner', 'upper_back', 'lower_back', 'left_haunch', 'right_haunch']
class policies: [{'id': 0, 'name': 'mouse', 'threshold': 0.25, 'track': True, 'keypoint_indices': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18]}, {'id': 1, 'name': 'landmarks', 'threshold': 0.25, 'track': False, 'keypoint_indices': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18]}]
expected output: [1, 300, 63]


In [4]:
# Export an FP32 ONNX model on CPU, then build the FP16 TensorRT engine on GPU.
import subprocess

if FINAL_PACKAGE_DIR.exists() and not OVERWRITE_EXISTING:
    raise FileExistsError(
        f"Existing package was preserved: {FINAL_PACKAGE_DIR}"
    )

# Remove a staging directory left by an earlier execution of this notebook cell.
previous_staging = globals().get("PACKAGE_DIR")
if (
    previous_staging
    and Path(previous_staging).parent.name.startswith(
        f".{MODEL_NAME}.build-"
    )
):
    cleanup_staging_package(previous_staging)

PACKAGE_DIR = create_staging_package(MODELS_DIR, MODEL_NAME)
atexit.register(cleanup_staging_package, PACKAGE_DIR)

print("Building in hidden staging package:", PACKAGE_DIR)

weights_dir = PACKAGE_DIR / "weights"
onnx_dir = PACKAGE_DIR / "onnx"
engines_dir = PACKAGE_DIR / "engines"
labels_dir = PACKAGE_DIR / "labels"
lib_dir = PACKAGE_DIR / "lib"
configs_dir = PACKAGE_DIR / "configs"
validation_dir = PACKAGE_DIR / "validation"

for directory in (
    weights_dir,
    onnx_dir,
    engines_dir,
    labels_dir,
    lib_dir,
    configs_dir,
    validation_dir,
):
    directory.mkdir(parents=True, exist_ok=True)

artifact_stem = (
    f"{SOURCE_MODEL.stem}_{PRECISION}_b{BATCH_SIZE}"
)

packaged_model = weights_dir / SOURCE_MODEL.name
packaged_parser = lib_dir / PARSER_LIBRARY.name
onnx_path = onnx_dir / f"{artifact_stem}.onnx"
engine_path = engines_dir / f"{artifact_stem}.engine"

shutil.copy2(SOURCE_MODEL, packaged_model)
shutil.copy2(PARSER_LIBRARY, packaged_parser)

# Export ONNX on the CPU. This avoids the PyTorch/cuDNN conflict encountered
# when Ultralytics performs its export-time dry run on CUDA.
export_model = YOLO(str(packaged_model))

exported_onnx = Path(
    export_model.export(
        format="onnx",
        device="cpu",
        imgsz=IMAGE_SIZE,
        batch=BATCH_SIZE,
        dynamic=False,
        simplify=True,
        nms=False,
        opset=18,
    )
).resolve()

assert exported_onnx.is_file(), exported_onnx

shutil.move(str(exported_onnx), str(onnx_path))
assert onnx_path.is_file(), onnx_path

# Validate the exported ONNX graph and its fixed tensor shapes.
onnx_model = onnx.load(str(onnx_path))
onnx.checker.check_model(onnx_model)


def tensor_shape(value_info):
    return [
        int(dim.dim_value) if dim.dim_value else dim.dim_param
        for dim in value_info.type.tensor_type.shape.dim
    ]


input_shape = tensor_shape(onnx_model.graph.input[0])
output_shapes = [
    tensor_shape(output)
    for output in onnx_model.graph.output
]

assert input_shape == [
    BATCH_SIZE,
    3,
    IMAGE_SIZE,
    IMAGE_SIZE,
], input_shape

assert output_shapes == [EXPECTED_OUTPUT], output_shapes

input_name = onnx_model.graph.input[0].name

print("ONNX:", onnx_path)
print("ONNX input:", input_name, input_shape)
print("ONNX outputs:", output_shapes)

# The ONNX graph is static, so do not pass --shapes to trtexec.
# trtexec uses the RTX GPU to compile the TensorRT engine.
trtexec_command = [
    "/usr/bin/trtexec",
    f"--onnx={onnx_path}",
    f"--saveEngine={engine_path}",
    "--skipInference",
]

if PRECISION == "fp16":
    trtexec_command.append("--fp16")
elif PRECISION != "fp32":
    raise ValueError(
        f"Unsupported TensorRT precision: {PRECISION}"
    )

print("Building TensorRT engine:")
print(" ".join(map(str, trtexec_command)))

subprocess.run(trtexec_command, check=True)

assert engine_path.is_file(), engine_path

raw_plan = engine_path.read_bytes()
assert raw_plan, "The generated TensorRT engine is empty"

# Confirm that TensorRT can deserialize the generated engine.
trt_logger = trt.Logger(trt.Logger.ERROR)
runtime = trt.Runtime(trt_logger)
engine = runtime.deserialize_cuda_engine(raw_plan)

assert engine is not None, (
    f"TensorRT could not deserialize {engine_path}"
)

print("TensorRT engine:", engine_path)
print(
    "TensorRT engine size:",
    f"{engine_path.stat().st_size / (1024 * 1024):.1f} MiB",
)
print("TensorRT deserialization: PASS")

Building in hidden staging package: /workspace/SqueakView/models/.mousehouse.build-7nf124k6/mousehouse
Ultralytics 8.4.152 🚀 Python-3.12.3 torch-2.14.0+cu130 CPU (Intel Core i7-8700K 3.70GHz)
YOLO26n-pose summary (fused): 132 layers, 3,018,927 parameters, 0 gradients, 8.0 GFLOPs

PyTorch: starting from '/workspace/SqueakView/models/.mousehouse.build-7nf124k6/mousehouse/weights/mousehouse_best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 300, 63) (8.0 MB)

ONNX: starting export with onnx 1.22.0 opset 18...
ONNX: slimming with onnxslim 0.1.96...
ONNX: export success ✅ 2.0s, saved as '/workspace/SqueakView/models/.mousehouse.build-7nf124k6/mousehouse/weights/mousehouse_best.onnx' (11.9 MB)

Export complete (2.4s)
Results saved to /workspace/SqueakView/models/.mousehouse.build-7nf124k6/mousehouse/weights/mousehouse_best.onnx
Predict:         yolo predict task=pose model=/workspace/SqueakView/models/.mousehouse.build-7nf124k6/mousehouse/weights/mousehouse_best.onnx img

[09/14/2026-20:36:32] [W] Weakly-typed networks have been deprecated in TensorRT. You can use the AutoCast tool (https://nvidia.github.io/TensorRT-Model-Optimizer/guides/8_autocast.html) to convert the network to be strongly typed.


[09/14/2026-20:36:32] [I] Selected Device: NVIDIA GeForce RTX 4060 Ti
[09/14/2026-20:36:32] [I] Selected Device ID: 0
[09/14/2026-20:36:32] [I] Selected Device UUID: GPU-6074635b-7874-5965-d072-ad56798715a1
[09/14/2026-20:36:32] [I] Compute Capability: 8.9
[09/14/2026-20:36:32] [I] SMs: 34
[09/14/2026-20:36:32] [I] Device Global Memory: 16379 MiB
[09/14/2026-20:36:32] [I] Shared Memory per SM: 100 KiB
[09/14/2026-20:36:32] [I] Memory Bus Width: 128 bits (ECC disabled)
[09/14/2026-20:36:32] [I] Application Compute Clock Rate: 2.655 GHz
[09/14/2026-20:36:32] [I] Application Memory Clock Rate: 9.001 GHz
[09/14/2026-20:36:32] [I] 
[09/14/2026-20:36:32] [I] Note: The application clock rates do not reflect the actual clock rates that the GPU is currently running at.
[09/14/2026-20:36:32] [I] 
[09/14/2026-20:36:32] [I] TensorRT version: 10.16.0
[09/14/2026-20:36:32] [I] Loading standard plugins
[09/14/2026-20:36:32] [I] [TRT] [MemUsageChange] Init CUDA: CPU +0, GPU +0, now: CPU 37, GPU 1156 (

In [ ]:
# Export an NMS-free ONNX model on CPU.
# TensorRT/DeepStream will perform the GPU-specific engine build.
export_model = YOLO(str(packaged_model))

ultralytics_onnx = Path(
    export_model.export(
        format="onnx",
        device="cpu",
        imgsz=IMAGE_SIZE,
        batch=BATCH_SIZE,
        dynamic=False,
        simplify=True,
        nms=False,
    )
).resolve()

assert ultralytics_onnx.is_file(), ultralytics_onnx
shutil.move(str(ultralytics_onnx), str(onnx_path))

# Validate the ONNX graph before building the TensorRT engine.
onnx_model = onnx.load(str(onnx_path))
onnx.checker.check_model(onnx_model)


def tensor_shape(value_info):
    return [
        int(dim.dim_value) if dim.dim_value else dim.dim_param
        for dim in value_info.type.tensor_type.shape.dim
    ]


input_info = onnx_model.graph.input[0]
input_name = input_info.name
input_shape = tensor_shape(input_info)
output_shapes = [
    tensor_shape(output)
    for output in onnx_model.graph.output
]

assert input_shape == [
    BATCH_SIZE,
    3,
    IMAGE_SIZE,
    IMAGE_SIZE,
], input_shape

assert output_shapes == [EXPECTED_OUTPUT], output_shapes

print("ONNX input:", input_name, input_shape)
print("ONNX outputs:", output_shapes)

# Build a raw TensorRT plan using the TensorRT installation supplied
# with the DeepStream container.
trtexec_path = shutil.which("trtexec")
assert trtexec_path, "trtexec was not found on PATH"

shape_spec = (
    f"{input_name}:"
    f"{BATCH_SIZE}x3x{IMAGE_SIZE}x{IMAGE_SIZE}"
)

trtexec_command = [
    trtexec_path,
    f"--onnx={onnx_path}",
    f"--saveEngine={engine_path}",
    f"--shapes={shape_spec}",
    "--skipInference",
]

if PRECISION == "fp16":
    trtexec_command.append("--fp16")
elif PRECISION != "fp32":
    raise ValueError(f"Unsupported precision: {PRECISION}")

print("Building TensorRT engine:")
print(" ".join(map(str, trtexec_command)))

subprocess.run(trtexec_command, check=True)

assert engine_path.is_file(), engine_path
raw_plan = engine_path.read_bytes()
assert raw_plan, "The generated TensorRT plan is empty"

# Confirm TensorRT can deserialize the generated raw plan.
logger = trt.Logger(trt.Logger.ERROR)
runtime = trt.Runtime(logger)
engine = runtime.deserialize_cuda_engine(raw_plan)
assert engine is not None, "TensorRT could not deserialize the engine"

print("ONNX:", onnx_path)
print("DeepStream engine:", engine_path)
print("TensorRT deserialization: PASS")

In [ ]:
import subprocess

In [ ]:
# Export directly with Ultralytics. DATA_YAML is passed as data=.
if FINAL_PACKAGE_DIR.exists() and not OVERWRITE_EXISTING:
    raise FileExistsError(f"Existing package was preserved: {FINAL_PACKAGE_DIR}")
previous_staging = globals().get("PACKAGE_DIR")
if previous_staging and Path(previous_staging).parent.name.startswith(f".{MODEL_NAME}.build-"):
    cleanup_staging_package(previous_staging)
PACKAGE_DIR = create_staging_package(MODELS_DIR, MODEL_NAME)
atexit.register(cleanup_staging_package, PACKAGE_DIR)
print("Building in hidden staging package:", PACKAGE_DIR)
weights_dir = PACKAGE_DIR / "weights"
onnx_dir = PACKAGE_DIR / "onnx"
engines_dir = PACKAGE_DIR / "engines"
labels_dir = PACKAGE_DIR / "labels"
lib_dir = PACKAGE_DIR / "lib"
configs_dir = PACKAGE_DIR / "configs"
validation_dir = PACKAGE_DIR / "validation"
for directory in (weights_dir, onnx_dir, engines_dir, labels_dir, lib_dir, configs_dir, validation_dir):
    directory.mkdir(parents=True, exist_ok=True)

artifact_stem = f"{SOURCE_MODEL.stem}_{PRECISION}_b{BATCH_SIZE}"
packaged_model = weights_dir / SOURCE_MODEL.name
packaged_parser = lib_dir / PARSER_LIBRARY.name
onnx_path = onnx_dir / f"{artifact_stem}.onnx"
engine_path = engines_dir / f"{artifact_stem}.engine"
shutil.copy2(SOURCE_MODEL, packaged_model)
shutil.copy2(PARSER_LIBRARY, packaged_parser)
export_model = YOLO(str(packaged_model))
ultralytics_engine = Path(export_model.export(
    format="engine",
    device=DEVICE,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    dynamic=False,
    quantize=16 if PRECISION == "fp16" else 32,
    simplify=True,
    nms=False,
    data=str(DATA_YAML),
)).resolve()
exported_onnx = packaged_model.with_suffix(".onnx")
assert ultralytics_engine.is_file()
assert exported_onnx.is_file()

shutil.move(str(exported_onnx), onnx_path)

# Ultralytics prefixes its plan with a 4-byte JSON length and JSON metadata.
# DeepStream needs only the raw TensorRT plan that follows it.
with ultralytics_engine.open("rb") as source:
    metadata_size = int.from_bytes(source.read(4), "little", signed=True)
    assert 0 < metadata_size < 16 * 1024 * 1024
    engine_metadata = json.loads(source.read(metadata_size))
    raw_plan = source.read()
assert engine_metadata.get("end2end") is True, engine_metadata
assert engine_metadata.get("args", {}).get("nms") is False, engine_metadata
assert raw_plan, "The exported TensorRT plan is empty"
engine_path.write_bytes(raw_plan)
ultralytics_engine.unlink()

# Validate ONNX structure and the raw TensorRT plan before packaging.
onnx_model = onnx.load(str(onnx_path))
onnx.checker.check_model(onnx_model)
def tensor_shape(value_info):
    return [int(dim.dim_value) if dim.dim_value else dim.dim_param for dim in value_info.type.tensor_type.shape.dim]
input_shape = tensor_shape(onnx_model.graph.input[0])
output_shapes = [tensor_shape(output) for output in onnx_model.graph.output]
assert input_shape == [BATCH_SIZE, 3, IMAGE_SIZE, IMAGE_SIZE], input_shape
assert output_shapes == [EXPECTED_OUTPUT], output_shapes
runtime = trt.Runtime(trt.Logger(trt.Logger.ERROR))
assert runtime.deserialize_cuda_engine(raw_plan) is not None
print("ONNX:", onnx_path)
print("DeepStream engine:", engine_path)

In [5]:
# Write the schema-3 model manifest and schema-2 pose-runtime contract.
classes_path = labels_dir / "classes.txt"
keypoints_path = labels_dir / "labels.txt"
classes_path.write_text("\n".join(class_names) + "\n")
keypoints_path.write_text("\n".join(keypoint_names) + "\n")

config_path = configs_dir / f"{MODEL_NAME}.txt"
parser_path = f"../lib/{packaged_parser.name}"
network_mode = 2 if PRECISION == "fp16" else 0
config_path.write_text(f"""[property]
gpu-id=0
net-scale-factor=0.00392156862745098
model-color-format=0
onnx-file=../onnx/{onnx_path.name}
model-engine-file=../engines/{engine_path.name}
network-mode={network_mode}
network-type=0
infer-dims=3;{IMAGE_SIZE};{IMAGE_SIZE}
batch-size={BATCH_SIZE}
output-tensor-meta=1
num-detected-classes={len(class_names)}
labelfile-path=../labels/classes.txt
parse-bbox-func-name=NvDsInferParseYolo26Pose
custom-lib-path={parser_path}
cluster-mode=4
maintain-aspect-ratio=1
symmetric-padding=1
gie-unique-id=1
interval=0
process-mode=1

[class-attrs-all]
pre-cluster-threshold={CONFIDENCE_THRESHOLD}
topk=300
""")

output_layer = onnx_model.graph.output[0].name
pose_schema = {
    "schema_version": 2,
    "task": "pose",
    "postprocess": "pyservicemaker_yolo26_pose_v1",
    "output_layer": output_layer,
    "input_width": IMAGE_SIZE,
    "input_height": IMAGE_SIZE,
    "letterbox": "symmetric",
    "end2end": True,
    "keypoint_labels_path": "../labels/labels.txt",
    "keypoint_count": keypoint_count,
    "keypoint_dims": keypoint_dims,
    "keypoint_threshold": KEYPOINT_THRESHOLD,
    "classes": pose_classes,
}
pose_path = configs_dir / f"{MODEL_NAME}.pose.json"
pose_path.write_text(json.dumps(pose_schema, indent=2) + "\n")

try:
    data_reference = DATA_YAML.resolve().relative_to(WORKSPACE).as_posix()
except ValueError:
    data_reference = DATA_YAML.name
dataset_sha256 = hashlib.sha256(DATA_YAML.read_bytes()).hexdigest()
def file_sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()
def artifact_identity(path):
    return {
        "path": path.relative_to(PACKAGE_DIR).as_posix(),
        "sha256": file_sha256(path),
    }
report_path = validation_dir / "import_report.json"
engine_execution = run_trtexec_validation(engine_path)
assert engine_execution["passed"], engine_execution
build_environment = {
    "tensorrt_version": str(trt.__version__),
    "cuda_version": str(torch.version.cuda),
    "gpu_name": str(torch.cuda.get_device_name(DEVICE)),
    "compute_capability": list(torch.cuda.get_device_capability(DEVICE)),
    "device_model": Path("/proc/device-tree/model").read_text(errors="replace").replace("\x00", "").strip(),
    "jetson_linux_release": Path("/etc/nv_tegra_release").read_text(errors="replace").strip(),
}
report_path.write_text(json.dumps({
    "onnx_input_shape": input_shape,
    "onnx_output_shapes": output_shapes,
    "output_layer": output_layer,
    "ultralytics_engine_metadata": engine_metadata,
    "build_environment": build_environment,
    "checks": {
        "onnx": True, "raw_engine": True, "yaml_labels": True, "schema_v2": True, "engine_identity": True, "engine_execution": True,
    },
    "engine_execution": engine_execution,
}, indent=2) + "\n")

artifact_identities = {
    "config": artifact_identity(config_path),
    "pose_schema": artifact_identity(pose_path),
    "onnx": artifact_identity(onnx_path),
    "engine": artifact_identity(engine_path),
    "class_labels": artifact_identity(classes_path),
    "keypoint_labels": artifact_identity(keypoints_path),
    "custom_parser": artifact_identity(packaged_parser),
    "import_report": artifact_identity(report_path),
}
manifest_path = PACKAGE_DIR / "model.yaml"
manifest_path.write_text(yaml.safe_dump({
    "schema_version": 3,
    "name": MODEL_NAME,
    "framework": "yolo26",
    "task": "pose",
    "precision": PRECISION,
    "batch_size": BATCH_SIZE,
    "classes": class_names,
    "keypoints": keypoint_names,
    "artifacts": artifact_identities,
    "export": {
        "builder": "ultralytics", "data": data_reference,
        "data_sha256": dataset_sha256, "end2end": True,
    },
}, sort_keys=False))

required = [packaged_model, packaged_parser, onnx_path, engine_path, classes_path, keypoints_path, config_path, pose_path, manifest_path, report_path]
assert all(path.is_file() for path in required)
assert pose_schema["schema_version"] == 2
assert yaml.safe_load(manifest_path.read_text())["schema_version"] == 3
assert pose_schema["classes"] == pose_classes
validate_model_package(config_path)
published_config = promote_model_package(
    PACKAGE_DIR, FINAL_PACKAGE_DIR,
    config_name=config_path.name, overwrite=OVERWRITE_EXISTING,
)
print("Package ready:", FINAL_PACKAGE_DIR)
print("Select config:", published_config)


FileNotFoundError: [Errno 2] No such file or directory: '/proc/device-tree/model'